In [1]:
import os
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyOAuth

load_dotenv()

auth_manager = SpotifyOAuth(
    client_id=os.getenv("CLIENT_ID"),
    client_secret=os.getenv("CLIENT_SECRET"),
    redirect_uri="https://127.0.0.1:8888/callback",
    scope="playlist-read-private playlist-read-collaborative user-library-read",
    open_browser=True
)

auth_url = auth_manager.get_authorize_url()
print("Go to this URL and approve access:")
print(auth_url)

Go to this URL and approve access:
https://accounts.spotify.com/authorize?client_id=e41504a74845451f9f7c81ec9722e2fd&response_type=code&redirect_uri=https%3A%2F%2F127.0.0.1%3A8888%2Fcallback&scope=playlist-read-private+playlist-read-collaborative+user-library-read


In [2]:
response_url = input("Paste the full redirect URL here: ")

code = auth_manager.parse_response_code(response_url)

token_info = auth_manager.get_access_token(code)

print("Access token acquired!")

Access token acquired!


C:\Users\cjwen\AppData\Local\Temp\ipykernel_17972\3933747703.py:5: DeprecationWarning: You're using 'as_dict = True'.get_access_token will return the token string directly in future versions. Please adjust your code accordingly, or use get_cached_token instead.
  token_info = auth_manager.get_access_token(code)


In [3]:
sp = spotipy.Spotify(auth=token_info["access_token"])

playlist_id = "6bkdfuvKMJPakfGBWmWHKM"

results = sp.playlist_items(playlist_id)
print(len(results["items"]))

40


In [4]:
tracks = []
results = sp.playlist_items(playlist_id, limit=100, offset=0)

tracks.extend(results['items'])

while results['next']:
    results = sp.next(results)  # fetch the next page
    tracks.extend(results['items'])

print(f"Collected {len(tracks)} tracks from playlist {playlist_id}")

Collected 40 tracks from playlist 6bkdfuvKMJPakfGBWmWHKM


In [5]:
import pandas as pd

track_list = []

for entry in tracks:
    track = entry.get("item")
    if track is None:
        continue  # skip unavailable tracks
    track_list.append({
        "track_id": track["id"],
        "name": track["name"],
        "artist": ", ".join([a["name"] for a in track["artists"]])
    })

df_tracks = pd.DataFrame(track_list)

# Save CSV of the private track IDs (your current playlist tracks)
df_tracks.to_csv("track_ids_private.csv", index=False)
print(f"Saved {len(df_tracks)} private track IDs to track_ids_private.csv")

Saved 40 private track IDs to track_ids_private.csv


In [6]:
public_tracks = []

for entry in tracks:  # 'tracks' is your list of playlist items from the user token
    track = entry.get("item")
    if track is None:
        continue
    
    track_name = track["name"]
    artist_name = track["artists"][0]["name"]  # take the first artist
    
    # Search for the track in the public catalog
    results = sp.search(q=f'track:{track_name} artist:{artist_name}', type='track', limit=1)
    items = results['tracks']['items']
    
    if len(items) == 0:
        # No public version found, skip this track
        continue
    
    public_track = items[0]
    
    public_tracks.append({
        "original_track_id": track["id"],       # private playlist ID
        "track_name": track_name,
        "artist": artist_name,
        "public_track_id": public_track["id"]   # global/public ID
    })

In [7]:
import pandas as pd

df_public_tracks = pd.DataFrame(public_tracks)
print(f"Found {len(df_public_tracks)} public/global track IDs out of {len(tracks)} total tracks")
df_public_tracks.head()

Found 40 public/global track IDs out of 40 total tracks


,original_track_id,track_name,artist,public_track_id
0,6fPan2saHdFaIHuTSatORv,L’AMOUR DE MA VIE,Billie Eilish,6fPan2saHdFaIHuTSatORv
1,0WbMK4wrZ1wFSty9F7FCgu,"Good Luck, Babe!",Chappell Roan,5sHuDRO059ES5fCMaP5Yql
2,34sOdxWu9FljH84UXdRwu1,all-american bitch,Olivia Rodrigo,34sOdxWu9FljH84UXdRwu1
3,4t2OeILB07eMGTXSUbMPEu,Oxytocin,Billie Eilish,4t2OeILB07eMGTXSUbMPEu
4,7e8utCy2JlSB8dRHKi49xM,Fluorescent Adolescent,Arctic Monkeys,7e8utCy2JlSB8dRHKi49xM


In [8]:
df_public_tracks.to_csv("public_track_ids.csv", index=False)
print("Saved public_track_ids.csv")

Saved public_track_ids.csv
